In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annocated, Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [ ]:
load_dotenv()

In [ ]:
model = ChatGoogleGenerativeAI(model="gpt-4o-mini", temperature=0.2, max_output_tokens=512)

In [ ]:
class SentimentSchema(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description = "Sentiment of the review")

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(
        description="The category of issue mentioned in the review"
    )
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(
        description="The emotional tone expressed by the user"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent or critical the issue appears to be"
    )

In [ ]:
Structred_model = model.with_structured_output(SentimentSchema)
Structred_model2 = model.with_structured_output(DiagnosisSchema)

In [ ]:
prompt = 'What is the sentiment of the following review - the software is very good'
structured_output = Structred_model.invoke(prompt)

In [ ]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive', 'negative']
    diagnosis: dict
    response: str

In [ ]:
def find_sentiment(state: ReviewState):
    prompt = f'for the folowing review find out the sentiment \n {state['review']}'
    sentiment = Structred_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}

In [ ]:
def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
    Also, kindly ask the user to leave feedback on our website."""

    response = model.invoke(prompt).content
    return {"response": response}

In [ ]:
def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
    """
    response = Structred_model2.invoke(prompt)

    return {"diagnosis": response.model_dump()}

In [ ]:
def negative_response(state: ReviewState):

    diagnosis = state["diagnosis"]

    prompt = f"""You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    Write an empathetic, helpful resolution message.
    """
    response = model.invoke(prompt).content

    return {"response": response}

In [ ]:
def check_sentiment(state :ReviewState) -> Literal['positive_response', 'run_diagnosis']:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node("positive_response", positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [ ]:
initial_State = {
    'review' : 'the product is realy worst'
}

workflow.invoke(initial_State)